In [103]:
import sys
sys.path.append('/pl/active/banich/studies/Relevantstudies/abcd/env/lib/python3.7/site-packages')

# mvpa for within-subject classification
# brainiak: https://brainiak.org/tutorials/
import os
import glob
import time
import numpy as np
import pandas as pd
import scipy.io
import nibabel as nib
from nilearn.maskers import NiftiMasker  # Updated import path for NiftiMasker
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import PredefinedSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.feature_selection import f_classif
from sklearn.svm import SVC
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime
from pingouin import ttest  # You may want to update pingouin to the latest version
from nilearn.image import new_img_like
import random
import pickle
import importlib as imp
import matplotlib.pylab as pylab
import matplotlib.patches as mpatches
from matplotlib.pyplot import figure

ImportError: cannot import name 'studentized_range' from 'scipy.stats' (/curc/sw/anaconda3/2020.11/lib/python3.8/site-packages/scipy/stats/__init__.py)

In [104]:
import glob
import pandas as pd
import nibabel as nib

def get_subjects():
    """
    Retrieves a list of subject numbers from the bids-hcp directory,
    excluding specific subjects.
    
    Returns:
        list: List of subject numbers as strings.
    """
    exclude_subjects = {'4004', '4020', '4035', '4037', '4052', '4061'}  # Define subjects to exclude
    subject_dirs = glob.glob('/pl/active/banich/studies/mindmem/analysis/bids-hcp/sub-*')
    subject_numbers = [sub.split('-')[-1] for sub in subject_dirs if sub.split('-')[-1] not in exclude_subjects]
    return sorted(subject_numbers)  # Sorting for better organization

In [105]:
def pull_norest_vols(sub, run, data_type, encoding=None, TR=0.46, hemodynamic_lag=0):
    """
    Retrieves non-resting volumes based on event timings for task and localizer data.
    
    Args:
        sub (str): Subject identifier.
        run (int): Run number.
        data_type (str): Type of data, either 'task' or 'localizer'.
        encoding (bool, optional): If True, filters for encoding trials in task data.
        TR (float): Repetition time of the scan in seconds.
        hemodynamic_lag (float): Hemodynamic lag to be applied to the onset times.
    
    Returns:
        tuple: Combined DataFrame with all files, data type, run, subject columns, and pulled volumes, and a list of errors.
    """
    data_frames = []
    error_subjects = []
    data_volume_indices = []
    path_to_func_bold = '/pl/active/banich/studies/mindmem/analysis/bids-hcp'
    
    if data_type == "task":
        file_pattern = f'/pl/active/banich/studies/mindmem/behavior/EVs/sub-{sub}/*/*/func-bold_task-WM*_dir-pa_run-0{run}_desc-stimulus_events.tsv'
        file_list = glob.glob(file_pattern)

        if not file_list:
            return pd.DataFrame(), [{'subject': sub, 'run': run, 'error': 'File not found'}]

        for file_path in file_list:
            try:
                files = pd.read_csv(file_path, sep=',')
                if files.shape[1] < 3:
                    files = pd.read_csv(file_path, sep='\t')
                
                if 'duration' not in files.columns:
                    raise KeyError("Missing 'duration' column")
                files['duration'] = pd.to_numeric(files['duration'], errors='coerce')
                files = files[np.isfinite(files['duration'])].copy()

                
                if encoding is not None:
                    files = files[files['trial_type'].str.contains('view_', na=False)].dropna()
                
                files['onset'] = pd.to_numeric(files['onset'], errors='coerce')
                files['duration'] = pd.to_numeric(files['duration'], errors='coerce')
                files.dropna(subset=['onset', 'duration'], inplace=True)
                files['adjusted_start'] = files['onset'] + hemodynamic_lag
                files['subject'] = sub
                files['run'] = run
                files['data_type'] = 'task'
                
                vol_times = files[~files['trial_type'].astype(str).str.contains("view", na=False)].copy()

                vol_times['stop'] = vol_times['adjusted_start'] + vol_times['duration']

                vol_times = vol_times[['adjusted_start', 'stop', 'duration']]
                vol_times.columns = ['start', 'stop', 'duration']

                # Ensure 'start_volume' and 'stop_volume' are integers
                vol_times['start_volume'] = np.floor(vol_times['start'] / TR).astype(int)
                vol_times['stop_volume'] = np.ceil(vol_times['stop'] / TR).astype(int)

                path_to_func_bold = '/pl/active/banich/studies/mindmem/analysis/bids-hcp'
                # Calculate max_volume by loading the BOLD file to get the total volume count
                bold_pattern = f'{path_to_func_bold}/sub-{sub}/ses-*/func/processed/sub-{sub}_ses-*_task-WM*_dir-pa_run-0{run}_space-MNI152NLin6Asym_desc-preproc_filtered_func_data_denoised_baseline_smoothed.nii.gz'
                bold_files = glob.glob(bold_pattern)
                if not bold_files:
                    raise FileNotFoundError(f"No BOLD file found for pattern: {bold_pattern}")

                bold_img = nib.load(bold_files[0])
                max_volume = bold_img.shape[-1]

                # Create a new column 'volumes_selected' that contains a list of volume indices between start and stop
                vol_times['volumes_selected'] = vol_times.apply(lambda row: list(range(int(row['start_volume']), min(int(row['stop_volume']), max_volume))), axis=1)


                # Flatten the 'volumes_selected' column and create a list of all volumes
                all_volumes = [vol for sublist in vol_times['volumes_selected'] for vol in sublist]

                # Remove duplicates, ensure no volume exceeds the maximum volume, and sort the volumes
                volume_indices = sorted(set([vol for vol in all_volumes if vol < max_volume]))
                volume_indices = pd.DataFrame(all_volumes, columns=['pul_vols']).assign(selector=run, sub=sub).reset_index(drop=True)
                
                
                data_volume_indices.append(volume_indices)
                data_frames.append(files)
            except Exception as e:
                error_subjects.append({'subject': sub, 'run': run, 'error': str(e)})
    
    if not data_frames:
        return pd.DataFrame(), pd.DataFrame(), error_subjects
    
    combined_df = pd.concat(data_frames, ignore_index=True)
    combined_vol_df = pd.concat(data_volume_indices, ignore_index=True)
    return combined_df, combined_vol_df, error_subjects

In [106]:
combined_df, combined_vol_df, error_subjects = pull_norest_vols('4001', 1, data_type='task', encoding=None, TR=0.46, hemodynamic_lag=0)

In [107]:
def process_all_subjects():
    """
    Processes all subjects across runs and data types.
    
    Returns:
        tuple: Combined DataFrame of all subjects and a list of errors encountered.
    """
    types = ['task']
    runs = range(1, 9)
    subs = get_subjects()
    sub_vols = []
    sub_all_vols = []
    error_log = []
    
    for s in subs:
        for r in runs:
            for t in types:
                result = pull_norest_vols(s, r, t, encoding=None, TR=0.46, hemodynamic_lag=0)

                if len(result) == 3:
                    df, vol_df, errors = result
                else:
                    df, vol_df, errors = pd.DataFrame(), pd.DataFrame(), []

                if not df.empty:
                    sub_vols.append(df)
                    sub_all_vols.append(vol_df)
                if errors:
                    error_log.extend(errors)

    final_df = pd.concat(sub_vols, ignore_index=True) if sub_vols else pd.DataFrame()
    vol_final_df = pd.concat(sub_all_vols, ignore_index=True) if sub_all_vols else pd.DataFrame()
    
    return final_df, vol_final_df, error_log

In [108]:
final_df, vol_final_df, error_log = process_all_subjects()

In [111]:
def create_task_stim(sub, run):
    global final_df

    file_pattern = (
        f"/pl/active/banich/studies/mindmem/behavior/EVs/sub-{sub}/*/*/"
        f"func-bold_task-WM*_dir-pa_run-0{run}_recording-pyschopy_stim.csv"
    )
    matches = glob.glob(file_pattern)
    if not matches:
        raise FileNotFoundError(f"No stim file found for sub={sub}, run={run}: {file_pattern}")
    file_path = matches[0]

    stim_files = pd.read_csv(file_path, sep=",")

    # Handle None values in 'fix_period_2.stopped'
    if "fix_period_2.stopped" in stim_files.columns and stim_files["fix_period_2.stopped"].eq("None").any():
        fix_cross_files = stim_files.loc[
            stim_files["fix_period_2.stopped"] != "None",
            ["fix_period_2.started", "fix_period_2.stopped"],
        ].copy()

        fix_cross_files = fix_cross_files.apply(pd.to_numeric, errors="coerce")
        avg_duration = (fix_cross_files["fix_period_2.stopped"] - fix_cross_files["fix_period_2.started"]).mean()

        stim_files.loc[stim_files["fix_period_2.stopped"] == "None", "fix_period_2.stopped"] = (
            pd.to_numeric(stim_files["fix_period_2.started"], errors="coerce") + avg_duration
        )

    # Ensure we have a key_resp.started anchor
    if "key_resp.started" not in stim_files.columns or stim_files["key_resp.started"].dropna().empty:
        raise ValueError(f"Missing or empty 'key_resp.started' for sub={sub}, run={run}")

    key0 = stim_files["key_resp.started"].dropna().reset_index(drop=True).iloc[0]

    stim_files["type"] = stim_files["OperationTop"] + "_" + stim_files["clusterID"] + "_" + stim_files["Valence"]
    stim_files["view"] = "view_" + stim_files["type"].str.extract(r"_(.*?)_")[0]
    stim_files["fix"] = "trial_fixation"

    def filter_nan(stim_file):
        stims_local = stim_file[["view", "type", "fix"]].copy()
        mask = stims_local[["view", "type"]].isna().all(axis=1)
        stims_local["group"] = (~mask).cumsum()
        stims_local = stims_local.loc[stims_local["group"] != 0].drop(columns=["group"])
        return stims_local.reset_index(drop=True)

    stims = filter_nan(stim_files)

    # Pull trial types from final_df, excluding start/end block markers
    trial_types = final_df.loc[
        (final_df["subject"] == sub)
        & (final_df["run"] == run)
        & (~final_df["trial_type"].astype(str).str.contains(r"view|ix", na=False))
        & (~final_df["trial_type"].isin(["start_block", "end_block"])),
        "trial_type",
    ].reset_index(drop=True)

    if trial_types.empty:
        raise ValueError(f"No trial_types found in final_df for sub={sub}, run={run}")

    if len(stims) != len(trial_types):
        raise ValueError(
            f"Length mismatch sub={sub}, run={run}: stims={len(stims)} vs trial_types={len(trial_types)}"
        )

    stims["trial_type"] = trial_types
    stims["view"] = trial_types
    stims["view"] = stims["view"].str.replace(r"^[^_]+_", "view_", regex=True)

    stims[["prefix", "valence"]] = stims["type"].str.rsplit("_", n=1, expand=True)
    stims = stims.drop(columns=["prefix", "type"])

    # Assumes valence is constant within run; preserves your original behavior
    stims["trial_type"] = stims["trial_type"] + "_" + stims["valence"].iloc[0]
    stims = stims[["view", "trial_type", "fix"]]

    new_cols = stims[["view", "trial_type", "fix"]].dropna().reset_index(drop=True).to_numpy().flatten()

    pull_cols = [
        "fix_period_2.started",
        "fix_period_2.stopped",
        "encode_item.started",
        "encode_item.stopped",
        "top_text.started",
        "top_text.stopped",
        "bottom_text.started",
        "bottom_text.stopped",
    ]

    missing = [c for c in pull_cols if c not in stim_files.columns]
    if missing:
        raise KeyError(f"Missing columns in stim file for sub={sub}, run={run}: {missing}")

    stim_times = stim_files[pull_cols].dropna().apply(pd.to_numeric, errors="coerce")
    stim_times = stim_times - key0

    stim_times["item_onset"] = stim_times["encode_item.started"]
    stim_times["fix_cross_onset"] = stim_times["fix_period_2.started"]
    stim_times["operation_onset"] = stim_times["top_text.started"]

    stim_times["item_duration"] = stim_times["encode_item.stopped"] - stim_times["encode_item.started"]
    stim_times["fix_cross_duration"] = stim_times["fix_period_2.stopped"] - stim_times["fix_period_2.started"]
    stim_times["operation_duration"] = stim_times["top_text.stopped"] - stim_times["top_text.started"]

    onset_cols = ["item_onset", "operation_onset", "fix_cross_onset"]
    duration_cols = ["item_duration", "operation_duration", "fix_cross_duration"]

    stim_times = stim_times[onset_cols + duration_cols]

    onset_melted = stim_times[onset_cols].to_numpy().flatten()
    duration_melted = stim_times[duration_cols].to_numpy().flatten()

    stim_times = pd.DataFrame(np.concatenate([[onset_melted], [duration_melted]])).T
    stim_times.columns = ["onset", "duration"]

    stim_times["trial_type"] = ["view_item", "operation", "fix_cross"] * int(stim_times.shape[0] / 3)
    stim_times["trial_type"] = new_cols

    stim_times["valence"] = stim_times["trial_type"].str.extract(r"_(positive|negative)$")[0].dropna().iloc[0]
    stim_times["trial_type"] = stim_times["trial_type"].str.replace(r"_(positive|negative)$", "", regex=True)

    stim_times = stim_times.assign(selector=run, subject=sub)
    stim_times = stim_times[["subject", "selector", "onset", "duration", "trial_type", "valence"]]

    return stim_times


def get_task_stim(sub):
    stims = [create_task_stim(sub, i) for i in range(1, 9)]
    return pd.concat(stims, ignore_index=True).sort_values(["selector", "onset"])


In [112]:
subs = get_subjects()
subs

['4001',
 '4002',
 '4003',
 '4005',
 '4006',
 '4007',
 '4008',
 '4009',
 '4010',
 '4011',
 '4012',
 '4013',
 '4014',
 '4015',
 '4016',
 '4017',
 '4018',
 '4019',
 '4021',
 '4022',
 '4023',
 '4024',
 '4025',
 '4026',
 '4027',
 '4028',
 '4029',
 '4030',
 '4031',
 '4032',
 '4033',
 '4034',
 '4036',
 '4038',
 '4039',
 '4040',
 '4041',
 '4042',
 '4043',
 '4044',
 '4045',
 '4046',
 '4047',
 '4048',
 '4049',
 '4050',
 '4051',
 '4053',
 '4054',
 '4055',
 '4056',
 '4057',
 '4058',
 '4059',
 '4060',
 '4062',
 '4063',
 '4064',
 '4065',
 '4066',
 '4067']

In [113]:
# Process all subjects, skipping errors and recording them
sub_stims = []
errors = []

for i in sorted(subs):
    try:
        sub_stims.append(get_task_stim(i))
    except Exception as e:
        errors.append({'subject_id': i, 'error_message': str(e)})
        
# Display errors if any
if errors:
    error_df = pd.DataFrame(errors)
if len(errors) < 1:
    print('No errors!')

In [114]:
#sub_stims

In [115]:
def pull_final_vols(sub_stim_df, sub, data_type, TR=0.46, hemodynamic_lag=0):
    """
    Pulls 30 TRs starting from each view_* trial, assigning those volumes
    to that view + the next 2 trials (op + fixation), preserving sequence.
    """
    import glob
    import numpy as np
    import nibabel as nib
    import pandas as pd

    trial_blocks = []
    volume_blocks = []
    error_subjects = []
    path_to_func_bold = '/pl/active/banich/studies/mindmem/analysis/bids-hcp'

    for run in range(1, 9):
        try:
            df = sub_stim_df.query('subject == @sub and selector == @run').copy()
            if df.empty:
                continue

            df['onset'] = pd.to_numeric(df['onset'], errors='coerce')
            df.dropna(subset=['onset'], inplace=True)
            df = df.sort_values('onset').reset_index(drop=True)
            df['adjusted_onset'] = df['onset'] + hemodynamic_lag

            # Load BOLD file
            bold_pattern = f'{path_to_func_bold}/sub-{sub}/ses-*/func/processed/sub-{sub}_ses-*_task-WM*_dir-pa_run-0{run}_space-MNI152NLin6Asym_desc-preproc_filtered_func_data_denoised_baseline_smoothed.nii.gz'
            bold_files = glob.glob(bold_pattern)
            if not bold_files:
                raise FileNotFoundError(f"No BOLD file found for pattern: {bold_pattern}")
            bold_img = nib.load(bold_files[0])
            max_volume = bold_img.shape[-1]

            block_id = 1
            i = 0
            while i < len(df):
                trial = df.iloc[i]
                if not trial['trial_type'].startswith('view'):
                    i += 1
                    continue

                # Collect view + operation + fixation
                block = df.iloc[i:i+3].copy()
                if len(block) < 3:
                    break

                view_onset = trial['adjusted_onset']
                start_vol = int(np.floor(view_onset / TR))
                stop_vol = min(start_vol + 30, max_volume)
                volume_list = list(range(start_vol, stop_vol))

                block['start_volume'] = start_vol
                block['stop_volume'] = stop_vol
                block['volumes_selected'] = [volume_list] * len(block)
                block['seq_itr'] = block_id
                block['sub'] = sub
                block['run'] = run

                trial_blocks.append(block)

                exploded = block[['subject', 'sub', 'run', 'selector', 'trial_type', 'valence', 'seq_itr', 'volumes_selected']].explode('volumes_selected')
                exploded = exploded.rename(columns={'volumes_selected': 'pul_vols'})
                volume_blocks.append(exploded)

                block_id += 1
                i += 3  # jump to next trial block

        except Exception as e:
            error_subjects.append({'subject': sub, 'run': run, 'error': str(e)})

    if not trial_blocks:
        return pd.DataFrame(), pd.DataFrame(), error_subjects

    final_df = pd.concat(trial_blocks, ignore_index=True)
    final_vol_df = pd.concat(volume_blocks, ignore_index=True)

    return final_df, final_vol_df, error_subjects


In [116]:
def process_final_subjects(sub_stim_df):
    """
    Processes all subjects across runs and data types.
    
    Returns:
        tuple: Combined DataFrame of all subjects and a list of errors encountered.
    """
    types = ['task']
    subs = get_subjects()
    sub_vols = []
    sub_all_vols = []
    error_log = []
    
    for s in subs:
        for t in types:
            result = pull_final_vols(sub_stim_df, s, t, TR=0.46, hemodynamic_lag=0)

            if len(result) == 3:
                df, vol_df, errors = result
            else:
                df, vol_df, errors = pd.DataFrame(), pd.DataFrame(), []

            if not df.empty:
                sub_vols.append(df)
            if not vol_df.empty:
                sub_all_vols.append(vol_df)
            if errors:
                error_log.extend(errors)

    final_df = pd.concat(sub_vols, ignore_index=True) if sub_vols else pd.DataFrame()
    vol_final_df = pd.concat(sub_all_vols, ignore_index=True) if sub_all_vols else pd.DataFrame()
    
    return final_df, vol_final_df, error_log


In [117]:
#these next section of chunks are for diagnostic purposes - 

subs = get_subjects()
print("n_subs:", len(subs))
print("first few:", subs[:5])

n_subs: 61
first few: ['4001', '4002', '4003', '4005', '4006']


In [118]:
#diagnostics
error_df = pd.DataFrame(errors)
print(error_df['error_message'].value_counts().head(20))
print(error_df.head(10))

No trial_types found in final_df for sub=4007, run=3    1
Name: error_message, dtype: int64
  subject_id                                      error_message
0       4007  No trial_types found in final_df for sub=4007,...


In [119]:
#diagnostics
print(final_df.columns.tolist())

['onset', 'duration', 'trial_type', 'adjusted_start', 'subject', 'run', 'data_type']


In [120]:
#diagnostics
print("final_df shape:", final_df.shape)
print("vol_final_df shape:", vol_final_df.shape)
print("n error_log:", len(error_log))

final_df shape: (53570, 7)
vol_final_df shape: (296690, 3)
n error_log: 1


In [121]:
#diagnostics
err = pd.DataFrame(error_log)
print(err.head(10))
print(err['error'].value_counts().head(30))

  subject  run                                              error
0    4007    3  No BOLD file found for pattern: /pl/active/ban...
No BOLD file found for pattern: /pl/active/banich/studies/mindmem/analysis/bids-hcp/sub-4007/ses-*/func/processed/sub-4007_ses-*_task-WM*_dir-pa_run-03_space-MNI152NLin6Asym_desc-preproc_filtered_func_data_denoised_baseline_smoothed.nii.gz    1
Name: error, dtype: int64


In [122]:
sub="4001"; run=1
ev = final_df[(final_df.subject==sub) & (final_df.run==run)].copy()

trial_types_dbg = ev.loc[
    (~ev["trial_type"].astype(str).str.contains(r"view|ix", na=False))
    & (~ev["trial_type"].isin(["start_block", "end_block"])),
    "trial_type"
].reset_index(drop=True)

print("trial_types_dbg len:", len(trial_types_dbg))
print(trial_types_dbg.value_counts())


trial_types_dbg len: 36
breath_face       5
maintain_place    5
track_place       5
suppress_face     5
suppress_place    4
track_face        4
breath_place      4
maintain_face     4
Name: trial_type, dtype: int64


In [123]:
#diagnostics
print("STIMS len:", len(stims))
print(stims.head(20))
print(stims.tail(20))

NameError: name 'stims' is not defined

In [124]:
#back to main analysis
final_df, final_vol_df, error_log = process_final_subjects(pd.concat(sub_stims))

In [126]:
final_processed_df = final_vol_df[
    ~final_vol_df["trial_type"].astype(str).str.contains(r"view|trial", na=False)
].copy()

In [127]:
import pandas as pd
import numpy as np

# Assuming `final_vol_df` contains the data
grouped_df = final_processed_df.dropna().groupby(['sub', 'selector']).count().reset_index()

# Compute summary statistics per selector
stats_df = grouped_df.groupby('selector')['pul_vols'].agg(['median', 'std']).reset_index()

# Merge with original grouped data
merged_df = grouped_df.merge(stats_df, on='selector')

# Calculate Z-score for anomaly detection
merged_df['z_score'] = (merged_df['pul_vols'] - merged_df['median']) / merged_df['std']

# Identify anomalies using a Z-score thresld (e.g., greater than 3 or less than -3)
anomalies = merged_df[np.abs(merged_df['z_score']) > 1.5]

from IPython.display import display, HTML

# Format sorted values into an HTML string with larger font
anomalies = sorted(anomalies['sub'].unique())
#display(HTML(f'<p style="white-space: nowrap; font-size: 20px; font-weight: bold;">' + ', '.join(map(str, anomalies)) + '</p>'))

In [128]:
problem_subjects = pd.DataFrame(anomalies, columns=['subject']).assign(missing_data=True)

good_subjects = pd.DataFrame([s for s in subs if s not in anomalies], columns=['subject']).assign(missing_data=False)
    
subject_status = pd.concat([good_subjects, problem_subjects]).sort_values('subject')

In [129]:
import pandas as pd
import numpy as np

def pad_missing_volumes(df, subject_status_df, expected_count=30):
    # Get list of subjects that need padding
    subjects_with_missing = subject_status_df.query("missing_data == True")['subject'].tolist()

    group_cols = ['subject', 'selector', 'seq_itr']
    id_cols = ['subject', 'sub', 'run', 'selector', 'trial_type', 'valence', 'seq_itr']
    
    padded_groups = []

    for group_keys, group_df in df.groupby(group_cols):
        subject = group_keys[0]

        # Only pad if the subject is marked as having missing data
        if subject in subjects_with_missing:
            n_missing = expected_count - len(group_df)
            if n_missing > 0:
                template = group_df.iloc[0][id_cols].copy()
                missing_df = pd.DataFrame([template.to_dict()] * n_missing)
                missing_df['pul_vols'] = np.NaN
                group_df = pd.concat([group_df, missing_df], ignore_index=True)

        padded_groups.append(group_df)

    return pd.concat(padded_groups, ignore_index=True)

# Example usage:
final_padded_df = pad_missing_volumes(final_processed_df, subject_status)


In [137]:
final_padded_df = final_padded_df.loc[:, ~final_padded_df.columns.duplicated()]

In [138]:
final_padded_df.columns[final_padded_df.columns.duplicated()]
# should now be empty

Index([], dtype='object')

In [139]:
final_padded_df = (
    final_padded_df
    .merge(subject_status, on="subject", how="left")
    .sort_values(["subject", "selector", "seq_itr", "pul_vols"])
    .rename(columns={"seq_itr": "identifier"})
)


KeyError: 'seq_itr'

In [ ]:
final_padded_df.to_csv(
    "/pl/active/banich/studies/mindmem/scripts/mvpa_scripts/process_data/sub_vol_trials_noshift.csv",
    index=False
)

In [140]:
print(final_padded_df.columns.tolist())

['subject', 'run', 'selector', 'trial_type', 'valence', 'identifier', 'pul_vols', 'missing_data']


In [141]:
# de-dupe columns just in case
final_padded_df = final_padded_df.loc[:, ~final_padded_df.columns.duplicated()].copy()

# OPTIONAL: if final_padded_df already has missing_data, don't merge it in again
subject_status2 = subject_status.drop(columns=["missing_data"], errors="ignore")

final_padded_df = (
    final_padded_df
    .merge(subject_status2, on="subject", how="left")
    .sort_values(["subject", "selector", "identifier", "pul_vols"])
)

final_padded_df.to_csv(
    "/pl/active/banich/studies/mindmem/scripts/mvpa_scripts/process_data/sub_vol_trials_noshift.csv",
    index=False
)


In [18]:
#subject_status.to_csv('/pl/active/banich/studies/Clearvale/scripts/jake_scripts/clearmem_processing/last_trial_problem_subs.csv', index=False)